# Signal with Mediation — SHAP, Causal SHAP and PCI

> **Summary.** Verifies all numerical values in Sections 4.4–4.6 of the paper
> for the chain model $X \to M \to Y$.  Computes plain SHAP, Causal SHAP, and
> PCI (with and without witnesses) for a fixed factual instance, then assembles
> the full desiderata table.

## Outline

1. Model and setup
2. Optimal predictors
3. Plain SHAP
4. Causal SHAP
5. PCI without witnesses
6. PCI with witnesses
7. Desiderata table and key findings

## 1. Model and setup

**SCM:** linear chain $X \to M \to Y$ with additive Gaussian noise.

| Variable | Distribution |
|---|---|
| $X$ | $\mathcal{N}(0.5,\; 0.25)$ |
| $M$ | $X + \varepsilon_M$, $\varepsilon_M \sim \mathcal{N}(0,\, 0.1)$ |
| $Y$ | $M + \varepsilon_Y$, $\varepsilon_Y \sim \mathcal{N}(0,\, 0.1)$ |

**Factual instance:** $x^\star = m^\star = y^\star = 1$ (both noise terms are zero).

**PCI contrast function:** $\mathrm{ci}(y^s, y^n, y^\star) = |y^n - y^\star| - |y^s - y^\star|$.
Since $y^s = y^\star = 1$ always, this reduces to $|y^n - 1|$.
Alternative $A' \sim P(A)$.

In [1]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(42)
N = 2_000_000

mu_X, var_X = 0.5, 0.25
var_eM = 0.1
var_eY = 0.1
var_M  = var_X + var_eM   # 0.35
var_Y  = var_M + var_eY   # 0.45
cov_XM = var_X            # 0.25
cov_XY = var_X            # 0.25
cov_MY = var_M            # 0.35
mu_M = mu_X; mu_Y = mu_X
x_star, m_star, y_star = 1.0, 1.0, 1.0

print(f"Var(X)={var_X}, Var(M)={var_M}, Var(Y)={var_Y}")
print(f"Cov(X,M)={cov_XM}, Cov(X,Y)={cov_XY}, Cov(M,Y)={cov_MY}")

Var(X)=0.25, Var(M)=0.35, Var(Y)=0.44999999999999996
Cov(X,M)=0.25, Cov(X,Y)=0.25, Cov(M,Y)=0.35


## 2. Optimal predictors

$f_Y(x, m) = m$ is exact.  $f_M$ and $f_X$ are the linear least-squares predictors
from the joint Gaussian (conditional expectations under the model distribution).

In [2]:
def f_Y(x, m): return m

Sigma_XY = np.array([[var_X, cov_XY],[cov_XY, var_Y]])
coeff_fM = np.array([cov_XM, cov_MY]) @ np.linalg.inv(Sigma_XY)
def f_M(x, y): return mu_M + coeff_fM[0]*(x - mu_X) + coeff_fM[1]*(y - mu_Y)

Sigma_MY = np.array([[var_M, cov_MY],[cov_MY, var_Y]])
coeff_fX = np.array([cov_XM, cov_XY]) @ np.linalg.inv(Sigma_MY)
def f_X(m, y): return mu_X + coeff_fX[0]*(m - mu_M) + coeff_fX[1]*(y - mu_Y)

print(f"f_M(X,Y) = 0.5 + {coeff_fM[0]:.4f}*(X-0.5) + {coeff_fM[1]:.4f}*(Y-0.5)  [expect 0.5X + 0.5Y]")
print(f"f_X(M,Y) = 0.5 + {coeff_fX[0]:.4f}*(M-0.5) + {coeff_fX[1]:.6f}*(Y-0.5)  [expect (5/7)*(M-0.5)]")
print(f"f_Y(1,1)={f_Y(1,1):.4f}  f_M(1,1)={f_M(1,1):.4f}  f_X(1,1)={f_X(1,1):.4f}  [1, 1, {5/14+0.5:.4f}]")

f_M(X,Y) = 0.5 + 0.5000*(X-0.5) + 0.5000*(Y-0.5)  [expect 0.5X + 0.5Y]
f_X(M,Y) = 0.5 + 0.7143*(M-0.5) + 0.000000*(Y-0.5)  [expect (5/7)*(M-0.5)]
f_Y(1,1)=1.0000  f_M(1,1)=1.0000  f_X(1,1)=0.8571  [1, 1, 0.8571]


## 3. Plain SHAP

Coalition values use conditional expectations: $v(S) = \mathbb{E}[B \mid X_S = x_S^\star]$,
marginalising missing features over their marginal distributions.  With $|N| = 2$
the Shapley formula reduces to

$$\phi_i = \tfrac{1}{2}\bigl(v(\{i\}) - v(\emptyset)\bigr)
          + \tfrac{1}{2}\bigl(v(N) - v(N \setminus \{i\})\bigr).$$

In [3]:
def E_cond(tmean, cov_to, var_o, oval, omean):
    return tmean + cov_to / var_o * (oval - omean)

def shapley2(vA, vB, vAB, v0):
    return 0.5*(vA-v0)+0.5*(vAB-vB), 0.5*(vB-v0)+0.5*(vAB-vA)

EY_X1 = E_cond(mu_Y, cov_XY, var_X, x_star, mu_X)
EX_Y1 = E_cond(mu_X, cov_XY, var_Y, y_star, mu_Y)
EM_Y1 = E_cond(mu_M, cov_MY, var_Y, y_star, mu_Y)

vY_e, vY_X, vY_M, vY_XM = mu_M, E_cond(mu_M, cov_XM, var_X, x_star, mu_X), m_star, 1.0
vM_e = f_M(mu_X, mu_Y)
vM_X = 0.5*x_star + 0.5*EY_X1
vM_Y = 0.5*EX_Y1 + 0.5*y_star
vM_XY = f_M(x_star, y_star)
vX_e = f_X(mu_M, mu_Y)
vX_M = f_X(m_star, mu_Y)
vX_Y = f_X(EM_Y1, y_star)
vX_MY = f_X(m_star, y_star)

phi_X_Y_plain, phi_M_Y_plain = shapley2(vY_X, vY_M, vY_XM, vY_e)
phi_X_M_plain, phi_Y_M_plain = shapley2(vM_X, vM_Y, vM_XY, vM_e)
phi_M_X_plain, phi_Y_X_plain = shapley2(vX_M, vX_Y, vX_MY, vX_e)

print("Plain SHAP:")
print(f"  phi_X^Y={phi_X_Y_plain:.4f} [0.250]  phi_M^Y={phi_M_Y_plain:.4f} [0.250]")
print(f"  phi_X^M={phi_X_M_plain:.4f} [0.306]  phi_Y^M={phi_Y_M_plain:.4f} [0.194]")
print(f"  phi_M^X={phi_M_X_plain:.4f} [0.219]  phi_Y^X={phi_Y_X_plain:.4f} [0.139]")

Plain SHAP:
  phi_X^Y=0.2500 [0.250]  phi_M^Y=0.2500 [0.250]
  phi_X^M=0.3056 [0.306]  phi_Y^M=0.1944 [0.194]
  phi_M^X=0.2183 [0.219]  phi_Y^X=0.1389 [0.139]


## 4. Causal SHAP

Coalition values use interventional expectations:
$v(S) = \mathbb{E}[B \mid \mathrm{do}(X_S = x_S^\star)]$.
Non-coalition features are sampled interventionally — descendants of $S$ from
their structural equations after the do-operator; non-descendants from their
marginals.

In [4]:
vcY_e, vcY_X, vcY_M, vcY_XM = mu_M, 1.0, 1.0, 1.0
vcM_e = f_M(mu_X, mu_Y)
vcM_X = 0.5*x_star + 0.5*1.0   # E[Y|do(X=1)] = 1
vcM_Y = 0.5*mu_X + 0.5*y_star  # E[X|do(Y=1)] = mu_X (breaks M->Y)
vcM_XY = f_M(x_star, y_star)
vcX_e = f_X(mu_M, mu_Y)
vcX_M = f_X(m_star, 1.0)        # do(M=1); Y coeff~0
vcX_Y = f_X(mu_M, y_star)       # do(Y=1) breaks M->Y; E[M|do(Y=1)]=mu_M
vcX_MY = f_X(m_star, y_star)

phi_X_Y_c, phi_M_Y_c = shapley2(vcY_X, vcY_M, vcY_XM, vcY_e)
phi_X_M_c, phi_Y_M_c = shapley2(vcM_X, vcM_Y, vcM_XY, vcM_e)
phi_M_X_c, phi_Y_X_c = shapley2(vcX_M, vcX_Y, vcX_MY, vcX_e)

print("Causal SHAP:")
print(f"  phi_X^Y={phi_X_Y_c:.4f} [0.250]  phi_M^Y={phi_M_Y_c:.4f} [0.250]")
print(f"  phi_X^M={phi_X_M_c:.4f} [0.375]  phi_Y^M={phi_Y_M_c:.4f} [0.125]")
print(f"  phi_M^X={phi_M_X_c:.4f} [0.357]  phi_Y^X={phi_Y_X_c:.4f} [0.000]")

Causal SHAP:
  phi_X^Y=0.2500 [0.250]  phi_M^Y=0.2500 [0.250]
  phi_X^M=0.3750 [0.375]  phi_Y^M=0.1250 [0.125]
  phi_M^X=0.3571 [0.357]  phi_Y^X=0.0000 [0.000]


## 5. PCI without witnesses

**Setup:** $\mathbf{S}$ = both input features of target $B$; $\mathbf{W}=\emptyset$
(so $T=\emptyset$ forced).  $\Gamma_s$ uniform over the 3 non-empty subsets of
$\mathbf{S}$; valid pairs have $C\ni A$ (weight $\tfrac{1}{3}$ each, two such
subsets giving total Γ-weight $\tfrac{2}{3}$).

Expected contrast $\mathbb{E}[|y^n - 1|]$ is computed in closed form via the
folded-normal formula, with Monte Carlo verification ($N = 2\times10^6$).

**Why values halve vs.\ single-suspect:** with $|C|=1$ and $|C|=2$ both weighted
$\tfrac{1}{3}$, the sum over $C\ni A$ is $\tfrac{2}{3}\times 0.677$, not $0.677$.

In [5]:
def E_abs_N(mu, sigma2):
    """E[|Z|] for Z ~ N(mu, sigma2) using the folded-normal formula."""
    sigma = np.sqrt(sigma2)
    c = mu / sigma
    return mu * (2*norm.cdf(c) - 1) + 2*sigma*norm.pdf(c)

X_alt  = rng.normal(mu_X, np.sqrt(var_X), N)
M_alt  = rng.normal(mu_M, np.sqrt(var_M), N)
eM_s   = rng.normal(0, np.sqrt(var_eM), N)
eY_s   = rng.normal(0, np.sqrt(var_eY), N)
eX_s   = rng.normal(0, np.sqrt(var_X),  N)   # for DXM C={X,Y} mc check

In [6]:
# S = {two input features of target}, W = empty.
# Valid (C, T) pairs: C in {A} and C in {A, other}, each with Gamma_s weight 1/3.
# Sufficiency term = 0 throughout (abducted factual noise, y^s = y*).
#
# DXY  S={X,M}: C={X}  do(X=X')         -> Y^n = X'+eM+eY  ~ N(mu_X, var_X+var_eM+var_eY)
#              C={X,M} do(X=X', M=M')   -> Y^n = M' +eY    ~ N(mu_M, var_M +var_eY)
#       Both give Y^n-1 ~ N(-0.5, 0.45). pci = (1/3+1/3) * E_abs_N(...)
e1 = E_abs_N(mu_X - y_star, var_X + var_eM + var_eY)   # C={X}
e2 = E_abs_N(mu_M - y_star, var_M + var_eY)             # C={X,M}: Y^n = M'+eY
pci_DXY_nw = (e1 + e2) / 3
mc_DXY_nw  = (np.mean(np.abs(X_alt + eM_s + eY_s - y_star))
              + np.mean(np.abs(M_alt + eY_s          - y_star))) / 3

# DMY  S={X,M}: C={M}  do(M=M')         -> Y^n = M' +eY  ~ N(-0.5, 0.45)
#              C={X,M} do(X=X', M=M')   -> Y^n = M' +eY  (same)
pci_DMY_nw = 2 * E_abs_N(mu_M - y_star, var_M + var_eY) / 3
mc_DMY_nw  = 2 * np.mean(np.abs(M_alt + eY_s - y_star)) / 3

# DXM  S={X,Y}: C={X}   do(X=X')        -> M^n = X'+eM  ~ N(-0.5, 0.35)
#               C={X,Y} do(X=X', Y=Y')  -> M^n = X'+eM  (Y not in M's eq)
pci_DXM_nw = 2 * E_abs_N(mu_X - m_star, var_X + var_eM) / 3
mc_DXM_nw  = 2 * np.mean(np.abs(X_alt + eM_s - m_star)) / 3

# DYX, DYM, DMX: suspect has no structural path to target -> X^n=X* or M^n=M* exactly.
pci_DYX_nw = pci_DYM_nw = pci_DMX_nw = 0.0

print("PCI without witnesses — S={two inputs}, W=∅ (closed-form | MC):")
print(f"  DXY: {pci_DXY_nw:.4f} | {mc_DXY_nw:.4f}  [expect 0.451]")
print(f"  DMY: {pci_DMY_nw:.4f} | {mc_DMY_nw:.4f}  [expect 0.451]")
print(f"  DXM: {pci_DXM_nw:.4f} | {mc_DXM_nw:.4f}  [expect 0.421]")
print(f"  DYX: {pci_DYX_nw:.4f}  DYM: {pci_DYM_nw:.4f}  DMX: {pci_DMX_nw:.4f}")
print()
print("DMXY without witnesses: DXY == DMY (exact tie) => FAILS")
print(f"  Both equal {pci_DXY_nw:.4f} because path-variance from X' to Y = "
      f"Var(X)+Var(eM)+Var(eY) = {var_X+var_eM+var_eY:.2f} = "
      f"Var(M)+Var(eY) = {var_M+var_eY:.2f}; tie holds for every C.")

PCI without witnesses — S={two inputs}, W=∅ (closed-form | MC):
  DXY: 0.4516 | 0.4515  [expect 0.451]
  DMY: 0.4516 | 0.4516  [expect 0.451]
  DXM: 0.4208 | 0.4205  [expect 0.421]
  DYX: 0.0000  DYM: 0.0000  DMX: 0.0000

DMXY without witnesses: DXY == DMY (exact tie) => FAILS
  Both equal 0.4516 because path-variance from X' to Y = Var(X)+Var(eM)+Var(eY) = 0.45 = Var(M)+Var(eY) = 0.45; tie holds for every C.


## 6. PCI with witnesses

**Setup:** $\mathbf{S}$ = both input features of target $B$; $\mathbf{W}$ = the
third variable (neither $A$ nor $B$). $\Gamma$ is built by rejection sampling
from marginals $\Gamma_s$ (uniform over the 3 non-empty subsets of $\mathbf{S}$)
and $\Gamma_w$ (uniform over $\{\emptyset,\{W\}\}$): draw $C\sim\Gamma_s$ and
$T\sim\Gamma_w$ independently, reject any draw with $T\cap C\neq\emptyset$.
The rejection step accepts 4 of the 6 raw pairs, giving normalizer
$Z=\tfrac{4}{6}=\tfrac{2}{3}$ and weight $p^\Gamma(C,T)=\tfrac{1}{4}$ on each
accepted pair.

The table below shows only the **active relevant** pairs — those that contain
the suspect $A$ and so contribute to $P_A^{s,n}$. (For each desideratum a
fourth pair is accepted by $\Gamma$ but does not contain $A$; it counts in
$Z$ but not in PCI.)

| desid | $C$ | $T$ | $p^\Gamma$ | $B^n$ |
|---|---|---|---|---|
| DXY ($A{=}X$, $W{=}M$) | $\{X\}$ | $\emptyset$ | 1/4 | $X'+\varepsilon_M+\varepsilon_Y$ |
| | $\{X\}$ | $\{M\}$ | 1/4 | $1+\varepsilon_Y$ (path severed) |
| | $\{X,M\}$ | $\emptyset$ | 1/4 | $M'+\varepsilon_Y$ |
| DMY ($A{=}M$, $W{=}X$) | $\{M\}$ | $\emptyset$ | 1/4 | $M'+\varepsilon_Y$ |
| | $\{M\}$ | $\{X\}$ | 1/4 | $M'+\varepsilon_Y$ ($X$ not in $Y$'s eq) |
| | $\{X,M\}$ | $\emptyset$ | 1/4 | $M'+\varepsilon_Y$ |
| DXM ($A{=}X$, $W{=}Y$) | $\{X\}$ | $\emptyset$ | 1/4 | $X'+\varepsilon_M$ |
| | $\{X\}$ | $\{Y\}$ | 1/4 | $X'+\varepsilon_M$ ($Y$ not in $M$'s eq) |
| | $\{X,Y\}$ | $\emptyset$ | 1/4 | $X'+\varepsilon_M$ |

Sufficiency term = 0 throughout (abducted factual noise gives $B^s=B^\star$).

In [7]:
# All weights are p^Gamma(C,T) = 1/4 under rejection-sampled Gamma (Z = 2/3).
# Only suspect-containing pairs contribute to PCI; non-suspect valid pairs
# count in Z but not here.

# ---------- DXY: S={X,M}, W={M} ----------
# (C={X}, T=∅)  1/4: do(X=X')        -> Y^n = X'+eM+eY     E = 0.677
# (C={X}, T={M}) 1/4: do(X=X', M=1)  -> Y^n = 1 + eY       E = sqrt(0.2/pi) ~ 0.252
# (C={X,M}, T=∅) 1/4: do(X=X', M=M') -> Y^n = M' + eY     E = 0.677
e_XY_free  = E_abs_N(mu_X - y_star, var_X + var_eM + var_eY)   # 0.677
e_XY_block = E_abs_N(0.0,           var_eY)                      # 0.252
e_XM_free  = E_abs_N(mu_M - y_star, var_M + var_eY)             # 0.677 (M' + eY)
pci_DXY_w  = (e_XY_free + e_XY_block + e_XM_free) / 4

mc_DXY_w = (np.mean(np.abs(X_alt + eM_s + eY_s - y_star))   # C={X}, T=∅
            + np.mean(np.abs(1.0   +         eY_s - y_star)) # C={X}, T={M}
            + np.mean(np.abs(M_alt +         eY_s - y_star)) # C={X,M}, T=∅
           ) / 4

# ---------- DMY: S={X,M}, W={X} ----------
# All three valid suspect-containing pairs give Y^n = M' + eY  ->  E = 0.677
# (C={M}, T=∅)   1/4: do(M=M')         Y^n=M'+eY
# (C={M}, T={X}) 1/4: do(M=M', X=1)    Y^n=M'+eY  (X not in Y's eq)
# (C={X,M}, T=∅) 1/4: do(X=X', M=M')   Y^n=M'+eY
pci_DMY_w  = 3 * E_abs_N(mu_M - y_star, var_M + var_eY) / 4
mc_DMY_w   = 3 * np.mean(np.abs(M_alt + eY_s - y_star)) / 4

# ---------- DXM: S={X,Y}, W={Y} ----------
# All three valid suspect-containing pairs give M^n = X' + eM  ->  E = 0.631
# (C={X}, T=∅)   1/4: do(X=X')         M^n=X'+eM
# (C={X}, T={Y}) 1/4: do(X=X', Y=1)    M^n=X'+eM  (Y not in M's eq)
# (C={X,Y}, T=∅) 1/4: do(X=X', Y=Y')   M^n=X'+eM
pci_DXM_w  = 3 * E_abs_N(mu_X - m_star, var_X + var_eM) / 4
mc_DXM_w   = 3 * np.mean(np.abs(X_alt + eM_s - m_star)) / 4

# ---------- zero desiderata ----------
# DYX: X exogenous, X^n=X*=1 always -> ci=0
# DYM: Y has no structural path to M (S={Y only}) -> ci=0
# DMX: X exogenous -> ci=0
pci_DYX_w = pci_DYM_w = pci_DMX_w = 0.0

print("PCI with witnesses — S={two inputs}, W={third variable} (closed-form | MC):")
print(f"  DXY (W=M): {pci_DXY_w:.4f} | {mc_DXY_w:.4f}  [expect 0.402]")
print(f"  DMY (W=X): {pci_DMY_w:.4f} | {mc_DMY_w:.4f}  [expect 0.508]")
print(f"  DXM (W=Y): {pci_DXM_w:.4f} | {mc_DXM_w:.4f}  [expect 0.473]")
print(f"  DYX: {pci_DYX_w:.4f}  DYM: {pci_DYM_w:.4f}  DMX: {pci_DMX_w:.4f}")
print()
print(f"DMXY with witnesses: DMY={pci_DMY_w:.3f} > DXY={pci_DXY_w:.3f}  => SATISFIED")
print("  Mechanism: M-witness case (weight 1/4) cuts X's path (0.252), lowering DXY")
print(f"  but X-witness has no effect on DMY (X not in Y's eq), leaving DMY = {pci_DMY_w:.3f}.")

PCI with witnesses — S={two inputs}, W={third variable} (closed-form | MC):
  DXY (W=M): 0.4018 | 0.4017  [expect 0.402]
  DMY (W=X): 0.5080 | 0.5080  [expect 0.508]
  DXM (W=Y): 0.4734 | 0.4730  [expect 0.473]
  DYX: 0.0000  DYM: 0.0000  DMX: 0.0000

DMXY with witnesses: DMY=0.508 > DXY=0.402  => SATISFIED
  Mechanism: M-witness case (weight 1/4) cuts X's path (0.252), lowering DXY
  but X-witness has no effect on DMY (X not in Y's eq), leaving DMY = 0.508.


## 7. Desiderata table and key findings

$\checkmark$ = desideratum satisfied; $\times$ = violated.

In [8]:
T = "✓"; X = "×"

def mark(v, ok_fn):
    return f"{v:.3f} {T if ok_fn(v) else X}"

def gt0(v): return v > 1e-9
def eq0(v): return abs(v) < 1e-9

rows = [
    ("DXY",  "R(X⇝Y) > 0",
     phi_X_Y_plain, phi_X_Y_c, pci_DXY_nw, pci_DXY_w, gt0),
    ("DMY",  "R(M⇝Y) > 0",
     phi_M_Y_plain, phi_M_Y_c, pci_DMY_nw, pci_DMY_w, gt0),
    ("DXM",  "R(X⇝M) > 0",
     phi_X_M_plain, phi_X_M_c, pci_DXM_nw, pci_DXM_w, gt0),
    ("DYX",  "R(Y⇝X) = 0",
     phi_Y_X_plain, phi_Y_X_c, pci_DYX_nw, pci_DYX_w, eq0),
    ("DYM",  "R(Y⇝M) = 0",
     phi_Y_M_plain, phi_Y_M_c, pci_DYM_nw, pci_DYM_w, eq0),
    ("DMX",  "R(M⇝X) = 0",
     phi_M_X_plain, phi_M_X_c, pci_DMX_nw, pci_DMX_w, eq0),
]

print(f"{'Desid':<5}  {'Condition':<20}  {'Plain':>9}  {'Causal':>9}  {'PCI W=∅':>9}  {'PCI W=3rd':>10}")
print("-" * 74)
for (d, cond, plain, causal, pci_nw, pci_w, ok) in rows:
    print(f"{d:<5}  {cond:<20}  {mark(plain,ok):>9}  {mark(causal,ok):>9}  "
          f"{mark(pci_nw,ok):>9}  {mark(pci_w,ok):>10}")

print()
print("DMXY  R(M⇝Y) > R(X⇝Y)")
print(f"  Plain:       {phi_M_Y_plain:.3f} = {phi_X_Y_plain:.3f}  {X}")
print(f"  Causal:      {phi_M_Y_c:.3f} = {phi_X_Y_c:.3f}  {X}")
print(f"  PCI W=∅:     {pci_DMY_nw:.3f} = {pci_DXY_nw:.3f}  {X}  (path-variances cancel)")
print(f"  PCI W=3rd:   {pci_DMY_w:.3f} > {pci_DXY_w:.3f}  {T}  (witness breaks X’s indirect path)")


Desid  Condition                 Plain     Causal    PCI W=∅   PCI W=3rd
--------------------------------------------------------------------------
DXY    R(X⇝Y) > 0              0.250 ✓    0.250 ✓    0.452 ✓     0.402 ✓
DMY    R(M⇝Y) > 0              0.250 ✓    0.250 ✓    0.452 ✓     0.508 ✓
DXM    R(X⇝M) > 0              0.306 ✓    0.375 ✓    0.421 ✓     0.473 ✓
DYX    R(Y⇝X) = 0              0.139 ×    0.000 ✓    0.000 ✓     0.000 ✓
DYM    R(Y⇝M) = 0              0.194 ×    0.125 ×    0.000 ✓     0.000 ✓
DMX    R(M⇝X) = 0              0.218 ×    0.357 ×    0.000 ✓     0.000 ✓

DMXY  R(M⇝Y) > R(X⇝Y)
  Plain:       0.250 = 0.250  ×
  Causal:      0.250 = 0.250  ×
  PCI W=∅:     0.452 = 0.452  ×  (path-variances cancel)
  PCI W=3rd:   0.508 > 0.402  ✓  (witness breaks X’s indirect path)


### Key findings

**Without witnesses** ($\mathbf{W} = \emptyset$): PCI satisfies the six
single-variable desiderata but **fails DMXY** — the same failure as both SHAP
variants.  Both DXY and DMY equal **0.452**, a tie driven by the path-variance
identity:
$\mathrm{Var}(X) + \mathrm{Var}(\varepsilon_M) + \mathrm{Var}(\varepsilon_Y) = 0.45
= \mathrm{Var}(M) + \mathrm{Var}(\varepsilon_Y)$.
Without pinning anything, PCI cannot distinguish $X$'s indirect effect from
$M$'s direct effect.

**With witnesses** ($\mathbf{W} =$ third variable): PCI satisfies all six
desiderata **and** DMXY.  Under the rejection-sampled $\Gamma$ (normalizer
$Z = 2/3$), three suspect-containing $(C,T)$ pairs each carry weight $1/4$.
For $R(X \rightsquigarrow Y)$, one of the three pins $M$ as an active witness,
severing $X$'s indirect path (contribution $\approx 0.252$); the other two
let $M$ propagate freely ($\approx 0.677$ each), giving
$\mathrm{PCI}(X \rightsquigarrow Y) \approx 0.402$.
For $R(M \rightsquigarrow Y)$ the $X$-witness has no effect (X does not appear
in $Y$'s structural equation), so all three cases contribute $\approx 0.677$
and $\mathrm{PCI}(M \rightsquigarrow Y) \approx 0.508 > 0.402$.
The witness mechanism is the ingredient that exposes the direct/indirect gap.